In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.silver;
CREATE SCHEMA IF NOT EXISTS main.gold;


In [0]:
%sql
DESCRIBE TABLE main.bronze.unplanned_hospital_visits;
SELECT DISTINCT measure_id FROM main.bronze.unplanned_hospital_visits WHERE upper(measure_id) LIKE '%HIP%';
SELECT DISTINCT measure_id FROM main.bronze.complications_and_deaths WHERE upper(measure_id) LIKE '%HIP%';

In [0]:
%sql
SELECT DISTINCT measure_id FROM main.bronze.unplanned_hospital_visits WHERE upper(measure_id) LIKE '%HIP%'

In [0]:
%sql

WITH dups AS (
  SELECT source_key, code_type, code, setting, billing_class, payer_name, plan_name, item_description,
         COUNT(*) AS n_rows,
         COUNT(DISTINCT negotiated_dollar) AS distinct_dollar_values
  FROM main.silver.silver_negotiated_rates
  GROUP BY ALL
  HAVING COUNT(*) > 1
)
SELECT source_key,
       COUNT(*) AS duplicate_groups,
       SUM(CASE WHEN distinct_dollar_values <= 1 THEN 1 ELSE 0 END) AS identical_amount_groups
FROM dups GROUP BY source_key

In [0]:
%sql
SELECT hospital_ccn, hospital_name, in_cms_general_info, has_cms_readmission_data,
       has_cms_complication_data, city, county, hospital_ownership, price_file_last_updated_on
FROM main.gold.dim_hospital

In [0]:
%sql
SELECT * FROM main.gold.gold_price_vs_readmission;
SELECT * FROM main.gold.gold_price_variation_across_hospitals ORDER BY code, setting;

In [0]:
-- cHecking silver tables mix professional (surgeon) fees with facility (hospital) fees.
SELECT source_key, code, setting, billing_class, COUNT(*) AS n_rates,
       MIN(negotiated_dollar) AS min_rate,
       percentile(negotiated_dollar, 0.5) AS median_rate,
       MAX(negotiated_dollar) AS max_rate
FROM main.silver.silver_negotiated_rates
WHERE code IN ('27447', '27130') AND rate_type = 'dollar' AND NOT is_placeholder_rate
GROUP BY ALL
ORDER BY code, source_key, setting, billing_class

In [0]:
WITH dup_rows AS (
  SELECT *, COUNT(*) OVER (PARTITION BY source_key, code_type, code, setting, billing_class,
                           payer_name, plan_name, item_description) AS n_in_group
  FROM main.silver.silver_negotiated_rates
),
numbered AS (
  SELECT *, DENSE_RANK() OVER (PARTITION BY source_key
                               ORDER BY code, payer_name, plan_name, setting, item_description) AS grp
  FROM dup_rows WHERE n_in_group > 1
)
SELECT source_key, grp, code, setting, billing_class, payer_name, plan_name, rate_type,
       negotiated_dollar, negotiated_percentage, methodology, gross_charge, min_charge, max_charge
FROM numbered WHERE grp <= 3
ORDER BY source_key, grp, negotiated_dollar

In [0]:
WITH dup_rows AS (
  SELECT *, COUNT(*) OVER (PARTITION BY source_key, code_type, code, setting, billing_class,
                           payer_name, plan_name, item_description) AS n_in_group
  FROM main.silver.silver_negotiated_rates
),
numbered AS (
  SELECT *, DENSE_RANK() OVER (PARTITION BY source_key
                               ORDER BY code, payer_name, plan_name, setting, item_description) AS grp
  FROM dup_rows WHERE n_in_group > 1
)
SELECT source_key, grp, code, setting, billing_class, payer_name, plan_name, rate_type,
       negotiated_dollar, negotiated_percentage, methodology, gross_charge, min_charge, max_charge
FROM numbered WHERE grp <= 3
ORDER BY source_key, grp, negotiated_dollar

In [0]:
SELECT COUNT(*) AS tn_hospitals_with_readmission_rate
FROM main.silver.stg_cms_outcomes o
JOIN main.silver.stg_cms_hospitals h ON o.hospital_ccn = h.hospital_ccn
WHERE h.state = 'TN' AND o.readmission_reported

In [0]:
SELECT source_key, code, setting, billing_class, rate_type, COUNT(*) AS n_rates
FROM main.silver.silver_negotiated_rates
WHERE code IN ('27447', '27130', '469', '470')
GROUP BY ALL
ORDER BY source_key, code, setting, billing_class, rate_type

In [0]:
SELECT item_description, item_codes, setting, COUNT(*) AS n,
       MIN(negotiated_dollar) AS min_rate, MAX(negotiated_dollar) AS max_rate
FROM main.silver.silver_negotiated_rates
WHERE source_key = 'vumc_mrf' AND code = '27130' AND rate_type = 'dollar'
GROUP BY ALL

In [0]:
SELECT payer_name, plan_name, COUNT(*) AS n
FROM main.silver.silver_negotiated_rates
WHERE rate_type = 'dollar'
GROUP BY ALL ORDER BY n DESC LIMIT 100

In [0]:
--test shows Which hospitals the duplicate groups come from, and what differs within them
WITH g AS (
  SELECT source_key, code_type, code, item_codes, modifiers, setting, billing_class,
         payer_name, plan_name, item_description,
         COUNT(*) AS n_rows,
         COUNT(DISTINCT negotiated_dollar) AS d_dollar,
         COUNT(DISTINCT methodology)       AS d_methodology,
         COUNT(DISTINCT gross_charge)      AS d_gross,
         COUNT(DISTINCT min_charge)        AS d_min,
         COUNT(DISTINCT max_charge)        AS d_max,
         COUNT(DISTINCT rate_type)         AS d_rate_type
  FROM main.silver.silver_negotiated_rates
  GROUP BY ALL
  HAVING COUNT(*) > 1
)
SELECT source_key,
       COUNT(*) AS dup_groups,
       SUM(CASE WHEN d_dollar > 1 THEN 1 ELSE 0 END)      AS different_dollar_amounts,
       SUM(CASE WHEN d_methodology > 1 THEN 1 ELSE 0 END) AS different_methodology,
       SUM(CASE WHEN d_gross > 1 OR d_min > 1 OR d_max > 1 THEN 1 ELSE 0 END) AS different_charges,
       SUM(CASE WHEN d_rate_type > 1 THEN 1 ELSE 0 END)   AS mixed_rate_types
FROM g
GROUP BY source_key
ORDER BY source_key

In [0]:
-- Actual exmple of a group with different dollar amounts
WITH dup_rows AS (
  SELECT *, COUNT(*) OVER (PARTITION BY source_key, code_type, code, item_codes, modifiers, setting,
                           billing_class, payer_name, plan_name, item_description) AS n_in_group
  FROM main.silver.silver_negotiated_rates
),
numbered AS (
  SELECT *, DENSE_RANK() OVER (PARTITION BY source_key
                               ORDER BY code, payer_name, plan_name, setting, item_description) AS grp
  FROM dup_rows WHERE n_in_group > 1
)
SELECT source_key, grp, code, setting, billing_class, payer_name, plan_name, rate_type,
       negotiated_dollar, negotiated_percentage, methodology, gross_charge, min_charge, max_charge
FROM numbered WHERE grp <= 3
ORDER BY source_key, grp, negotiated_dollar

In [0]:
--What VUMC’s rates look like?
SELECT code, setting, billing_class, rate_type, COUNT(*) AS n_rates
FROM main.silver.silver_negotiated_rates
WHERE source_key = 'vumc_mrf' AND code IN ('27447', '27130', '469', '470')
GROUP BY ALL
ORDER BY code, setting, billing_class, rate_type

In [0]:
-- Error in ingestion for vumc
   SELECT event_ts, status, note, error_message, etag, last_modified
   FROM main.bronze.ingestion_manifest
   WHERE dataset_key = 'vumc_mrf'
   ORDER BY event_ts DESC
   LIMIT 10

In [0]:
SELECT * FROM main.gold.gold_price_variation_across_hospitals
WHERE code IN ('27447', '27130') AND billing_class = 'facility'
ORDER BY code, setting

In [0]:
   SELECT * FROM main.gold.gold_price_variation_across_hospitals
   WHERE code IN ('27447', '27130') AND billing_class = 'facility'
   ORDER BY code, setting

In [0]:
   --Payer and Plan name
   SELECT payer_name, plan_name, COUNT(*) AS n
   FROM main.silver.silver_negotiated_rates
   WHERE rate_type = 'dollar'
   GROUP BY ALL ORDER BY n DESC LIMIT 150

In [0]:
SELECT payer_category, payer_category_is_default, COUNT(*) AS n_payer_plans
FROM main.silver.stg_payer_categories
GROUP BY ALL ORDER BY n_payer_plans DESC

In [0]:
SELECT * FROM main.gold.gold_hospital_comparison_by_payer_category
WHERE code IN ('27447', '27130') AND billing_class = 'facility' AND payer_category = 'commercial'
ORDER BY code, setting

In [0]:
SELECT hospital_name, payer_name, plan_name,
       MIN(negotiated_dollar) AS min_rate, MAX(negotiated_dollar) AS max_rate
FROM main.silver.silver_negotiated_rates
WHERE payer_category = 'commercial'
  AND code IN ('27447', '27130') AND billing_class = 'facility'
  AND rate_type = 'dollar' AND NOT is_placeholder_rate
GROUP BY ALL
ORDER BY min_rate
LIMIT 40

In [0]:
SELECT item_description, item_codes, setting, negotiated_dollar, methodology
FROM main.silver.silver_negotiated_rates
WHERE source_key = 'struth_mrf' AND code = '27447' AND rate_type = 'dollar'
  AND payer_name = 'CIGNA HMO' AND plan_name LIKE '3188%'
ORDER BY negotiated_dollar

In [0]:
SELECT item_description, modifiers, negotiated_dollar
FROM main.silver.silver_negotiated_rates
WHERE source_key = 'struth_mrf' AND code = '27447' AND rate_type = 'dollar'
  AND payer_name = 'CIGNA HMO' AND plan_name LIKE '3188%'
ORDER BY negotiated_dollar

In [0]:
SELECT * FROM main.gold.gold_hospital_comparison_by_payer_category
WHERE code IN ('27447', '27130') AND billing_class = 'facility' AND payer_category = 'commercial'
ORDER BY code, setting

In [0]:
SELECT COUNT(*) AS tn_hospitals_with_readmission_rate
FROM main.silver.stg_cms_outcomes o
JOIN main.silver.stg_cms_hospitals h ON o.hospital_ccn = h.hospital_ccn
WHERE h.state = 'TN' AND o.readmission_reported

In [0]:
%python
import httpx
from ingestion import config

In [0]:
%python
## checking what exactly does server return for VMC error
import httpx
from ingestion import config
url = next(s.url for s in config.HOSPITAL_FILES if s.key == "vumc_mrf")
with httpx.Client(headers={"User-Agent": config.FILES.user_agent}, follow_redirects=True) as c:
       with c.stream("GET", url, timeout=60) as r:
           print(r.status_code, r.headers.get("content-type"), r.headers.get("content-length"))
           print(r.headers.get("etag"), r.headers.get("last-modified"))
           print(r.url)
           print(next(r.iter_bytes(400)))

In [0]:
%python
url = "https://finance.vumc.org/assets/pub/pt/352528741_vanderbilt-university-medical-center_standardcharges.csv"
with httpx.Client(headers={"User-Agent": "Mozilla/5.0 (compatible; nss-de-final-project/1.0)"}, follow_redirects=True) as c:
    with c.stream("GET", url, timeout=60) as r:
        print(r.status_code, r.headers.get("content-type"), r.headers.get("content-length"))
        print("etag:", r.headers.get("etag"), "| last-modified:", r.headers.get("last-modified"))
        print(str(r.url)[:120])
        print(next(r.iter_bytes(4000)).decode("utf-8-sig", errors="ignore"))